In [2]:
import os
import pandas as pd

# 1. Load the raw nav_history file
NAV_PATH = os.path.join("..", "data", "raw", "02_nav_history.csv")
df_nav = pd.read_csv(NAV_PATH)

print(f"📋 Initial Rows in NAV History: {len(df_nav)}")
print("-" * 50)

# A. Parse dates to datetime
# dayfirst=True is crucial for Indian financial datasets typically formatted as DD-MM-YYYY
df_nav['date'] = pd.to_datetime(df_nav['date'], format= 'mixed', dayfirst=True)

📋 Initial Rows in NAV History: 46000
--------------------------------------------------


In [3]:
# B. Sort by amfi_code and date
df_nav = df_nav.sort_values(by=['amfi_code', 'date']).reset_index(drop=True)
print("✅ Dates parsed successfully using 'mixed' format strategy.")

✅ Dates parsed successfully using 'mixed' format strategy.


In [4]:
# C. Remove duplicate entries
duplicate_count = df_nav.duplicated(subset=['amfi_code', 'date']).sum()
df_nav = df_nav.drop_duplicates(subset=['amfi_code', 'date'], keep='first')
print(f"🧹 Removed {duplicate_count} duplicate rows.")

🧹 Removed 0 duplicate rows.


In [5]:
print("🔍 Total Null Values per Column:")
print(df_nav.isnull().sum())

🔍 Total Null Values per Column:
amfi_code    0
date         0
nav          0
dtype: int64


In [7]:
# E. Validate NAV > 0
initial_count = len(df_nav)
df_nav = df_nav[df_nav['nav'] > 0]
invalid_nav_count = initial_count - len(df_nav)
print(f"🛡️ Validated NAV > 0: Dropped {invalid_nav_count} rows.")
print("-" * 50)
print(f"🚀 Final Cleaned Rows in NAV History: {len(df_nav)}")

🛡️ Validated NAV > 0: Dropped 0 rows.
--------------------------------------------------
🚀 Final Cleaned Rows in NAV History: 46000


In [ ]:
processed_dir = os.path.join("..", "data", "processed")
processed_file_path = os.path.join(processed_dir, "02_nav_history_clean.csv")
df_nav.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned dataset successfully saved to: {processed_file_path}")


💾 Cleaned dataset successfully saved to: ..\data\processed\02_nav_history_clean.csv


In [9]:
import os
import pandas as pd

# 1. Load the raw transaction dataset
TXN_PATH = os.path.join("..", "data", "raw", "08_investor_transactions.csv")
df_txn = pd.read_csv(TXN_PATH)

print(f"📋 Initial Rows in Transactions Dataset: {len(df_txn)}")
print("\n🔍 Step 0: Checking for Null Values before cleaning:")
print(df_txn.isnull().sum())
print("-" * 60)

📋 Initial Rows in Transactions Dataset: 32778

🔍 Step 0: Checking for Null Values before cleaning:
investor_id           0
transaction_date      0
amfi_code             0
transaction_type      0
amount_inr            0
state                 0
city                  0
city_tier             0
age_group             0
gender                0
annual_income_lakh    0
payment_mode          0
kyc_status            0
dtype: int64
------------------------------------------------------------


In [10]:
# A. Fix Date Formats
# Identifying the date column dynamically (handles 'date', 'txn_date', 'transaction_date')
date_col = [col for col in df_txn.columns if 'date' in col.lower()][0]
df_txn[date_col] = pd.to_datetime(df_txn[date_col], format='mixed', dayfirst=True)
print(f"📅 Date column '{date_col}' successfully parsed to uniform datetime format.")

📅 Date column 'transaction_date' successfully parsed to uniform datetime format.


In [12]:
# B. Standardize Transaction Type
type_col = [col for col in df_txn.columns if 'type' in col.lower()][0]
# Convert to string, strip whitespace, and uppercase to make them perfectly uniform
df_txn[type_col] = df_txn[type_col].astype(str).str.strip().str.upper()
print(f"🔄 Transaction types in '{type_col}' standardized. Unique values now: {df_txn[type_col].unique()}")

🔄 Transaction types in 'transaction_type' standardized. Unique values now: ['SIP' 'REDEMPTION' 'LUMPSUM']


In [13]:
# D. Check KYC Status Values
kyc_col = [col for col in df_txn.columns if 'kyc_status' in col.lower()]
if kyc_col:
    kyc_col = kyc_col[0]
    print(f"\n👤 KYC STATUS VALUES DISTRIBUTION ('{kyc_col}'):")
    print(df_txn[kyc_col].value_counts(dropna=False))
else:
    print("\n⚠️ No explicit KYC column automatically detected.")

print("-" * 60)
print(f"🚀 Final Cleaned Rows in Transactions: {len(df_txn)}")


👤 KYC STATUS VALUES DISTRIBUTION ('kyc_status'):
kyc_status
Verified    30146
Pending      2632
Name: count, dtype: int64
------------------------------------------------------------
🚀 Final Cleaned Rows in Transactions: 32778


In [14]:
processed_dir = os.path.join("..", "data", "processed")

processed_file_path = os.path.join(processed_dir, "clean_transactions.csv")
df_txn.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned transaction dataset successfully saved to: {processed_file_path}")
print("=" * 60)


💾 Cleaned transaction dataset successfully saved to: ..\data\processed\clean_transactions.csv


In [15]:
import os
import pandas as pd
import numpy as np

# 1. Load the raw fund performance dataset
PERF_PATH = os.path.join("..", "data", "raw", "07_scheme_performance.csv")
df_perf = pd.read_csv(PERF_PATH)

print(f"📋 Initial Rows in Performance Dataset: {len(df_perf)}")
print("-" * 60)

📋 Initial Rows in Performance Dataset: 40
------------------------------------------------------------


In [16]:
# A. Isolate and validate return columns are numeric
return_cols = [col for col in df_perf.columns if 'return' in col.lower()]
print(f"🎯 Target Return Columns: {return_cols}")

initial_row_count = len(df_perf)
for col in return_cols:
    df_perf[col] = pd.to_numeric(df_perf[col], errors='coerce')

# Drop rows where return columns are invalid/NaN
df_perf = df_perf.dropna(subset=return_cols)
print(f"✅ Return values validated as numeric. Dropped {initial_row_count - len(df_perf)} invalid records.")

🎯 Target Return Columns: ['return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct']
✅ Return values validated as numeric. Dropped 0 invalid records.


In [17]:
# B. Check Expense Ratio Range (Strictly between 0.1% and 2.5%)
expense_col = [col for col in df_perf.columns if 'expense' in col.lower()][0]
# Ensure it is numeric first
df_perf[expense_col] = pd.to_numeric(df_perf[expense_col], errors='coerce')

before_expense_filter = len(df_perf)
# Filtering rows strictly within the 0.1% to 2.5% boundary
df_perf = df_perf[(df_perf[expense_col] >= 0.1) & (df_perf[expense_col] <= 2.5)]
print(f"📉 Checked Expense Ratio range [0.1% - 2.5%]: Filtered out {before_expense_filter - len(df_perf)} non-compliant rows.")

📉 Checked Expense Ratio range [0.1% - 2.5%]: Filtered out 0 non-compliant rows.


In [20]:
# C. Flag Negative Sharpe Ratios
sharpe_col = [col for col in df_perf.columns if 'sharpe' in col.lower()][0]
df_perf[sharpe_col] = pd.to_numeric(df_perf[sharpe_col], errors='coerce')

# 1 if negative, 0 if positive or zero
df_perf['is_underperforming_risk_free'] = np.where(df_perf[sharpe_col] < 0, 1, 0)
negative_sharpe_count = df_perf['is_underperforming_risk_free'].sum()
print(f"🚩 Flagged {negative_sharpe_count} funds with negative Sharpe ratios.")

print("-" * 60)
print(f"🚀 Final Cleaned Rows in Performance Catalog: {len(df_perf)}")

🚩 Flagged 0 funds with negative Sharpe ratios.
------------------------------------------------------------
🚀 Final Cleaned Rows in Performance Catalog: 40


In [21]:
processed_dir = os.path.join("..", "data", "processed")

processed_file_path = os.path.join(processed_dir, "clean_performance.csv")
df_perf.to_csv(processed_file_path, index=False)

print(f"\n💾 Cleaned dataset successfully saved to: {processed_file_path}")
print("=" * 60)


💾 Cleaned dataset successfully saved to: ..\data\processed\clean_performance.csv


In [1]:
import os
import pandas as pd

# 1. Define paths
raw_dir = os.path.join("..", "data", "raw")
processed_dir = os.path.join("..", "data", "processed")

# List of files we have ALREADY cleaned (we skip these so we don't overwrite our custom work!)
custom_cleaned_files = ["02_nav_history.csv", "08_investor_transactions.csv", "07_scheme_performance.csv"]

# 2. Get all remaining CSV files in the raw folder
all_raw_files = [f for f in os.listdir(raw_dir) if f.endswith('.csv')]
remaining_files = [f for f in all_raw_files if f not in custom_cleaned_files]

print(f"🔄 Found {len(remaining_files)} baseline datasets for general sanitization.\n")

# 3. Loop through and clean each remaining dataset
for file_name in remaining_files:
    raw_file_path = os.path.join(raw_dir, file_name)
    
    # Load dataset
    df = pd.read_csv(raw_file_path)
    initial_shape = df.shape
    
    # --- OPERATION A: Remove complete row duplicates ---
    df = df.drop_duplicates().reset_index(drop=True)
    
    # --- OPERATION B: Drop rows that are completely empty (all NaN values) ---
    df = df.dropna(how='all')
    
    # --- OPERATION C: Clean whitespace from text columns ---
    text_cols = df.select_dtypes(include=['object']).columns
    for col in text_cols:
        df[col] = df[col].astype(str).str.strip()
    
    final_shape = df.shape
    
    # Log the output metrics for the report
    print(f"📄 Dataset: {file_name}")
    print(f"   • Initial Matrix : {initial_shape[0]} rows, {initial_shape[1]} columns")
    print(f"   • Sanitized Matrix: {final_shape[0]} rows, {final_shape[1]} columns")
    print(f"   • Rows Removed   : {initial_shape[0] - final_shape[0]}")
    
    # --- OPERATION D: Export to Processed Folder ---
    clean_file_name = file_name.replace(".csv", "_clean.csv")
    output_path = os.path.join(processed_dir, clean_file_name)
    df.to_csv(output_path, index=False)
    print(f"   💾 Saved to: {output_path}\n" + "-"*50)

print("🎉 General data sanitization loop complete! All datasets are polished.")

🔄 Found 12 baseline datasets for general sanitization.

📄 Dataset: 01_fund_master.csv
   • Initial Matrix : 40 rows, 15 columns
   • Sanitized Matrix: 40 rows, 15 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\01_fund_master_clean.csv
--------------------------------------------------
📄 Dataset: 03_aum_by_fund_house.csv
   • Initial Matrix : 90 rows, 5 columns
   • Sanitized Matrix: 90 rows, 5 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\03_aum_by_fund_house_clean.csv
--------------------------------------------------
📄 Dataset: 04_monthly_sip_inflows.csv
   • Initial Matrix : 48 rows, 6 columns
   • Sanitized Matrix: 48 rows, 6 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed\04_monthly_sip_inflows_clean.csv
--------------------------------------------------
📄 Dataset: 05_category_inflows.csv
   • Initial Matrix : 144 rows, 3 columns
   • Sanitized Matrix: 144 rows, 3 columns
   • Rows Removed   : 0
   💾 Saved to: ..\data\processed

In [59]:
import os

# 1. Define the schema content
schema_content = """-- ====================================================================
-- STAR SCHEMA BLUEPRINT FOR MUTUAL FUND ANALYTICS DB
-- ====================================================================

-- DIMENSION 1:
CREATE TABLE dim_fund (
    amfi_code INTEGER PRIMARY KEY,
    fund_house TEXT NOT NULL,
    category TEXT,
    expense_ratio_pct REAL,
    min_sip_amount REAL,
    risk_category TEXT
);

-- DIMENSION 2: 
CREATE TABLE dim_date (
     date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT NOT NULL UNIQUE,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    is_weekday INTEGER
);

-- FACT 1: 
CREATE TABLE fact_nav (
    amfi_code INTEGER FOREIGN KEY,
    date INTEGER FOREIGN KEY,
    nav REAL NOT NULL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
    FOREIGN KEY (date) REFERENCES dim_date(date_id)
);

-- FACT 2:
CREATE TABLE fact_transactions (
    tx_id INTEGER PRIMARY KEY AUTOINCREMENT,
        investor_id INTEGER,
        amfi_code INTEGER,
        transaction_date TEXT,
        amount_inr REAL,
        transaction_type TEXT CHECK(transaction_type IN ('LUMPSUM', 'REDEMPTION', 'SIP')),
        state TEXT,
        city TEXT,
        city_tier VARCHAR(10),
        age_group VARCHAR(10),
        gender VARCHAR(10),
        FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code),
        FOREIGN KEY (transaction_date) REFERENCES dim_date(date)
);

-- FACT 3:
CREATE TABLE fact_performance (
    amfi_code INTEGER ,
    return_1yr_pct REAL,
    sharpe_ratio REAL,
    alpha REAL,
    max_drawdown_pct REAL,
    expense_ratio_pct REAL,
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- FACT 4: 
CREATE TABLE fact_portfolio (
    amfi_code INTEGER,
    stock_symbol TEXT,
    weight_pct REAL,
    sector TEXT,
    portfolio_date DATE
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);

-- FACT 5:
CREATE TABLE fact_aum (
    fund_house TEXT,
    date DATE,
    aum_crore REAL,
    num_schemes INTEGER
);

-- FACT 6:
CREATE TABLE fact_sip_industry (
    month INTEGER,
    sip_inflow_crore REAL,
    yoy_growth_pct REAL,
    active_sip_accounts_crore INTEGER
);
"""

# 2. Establish directory path and write down the schema.sql artifact
database_dir = os.path.join("..", "sql")

schema_file_path = os.path.join(database_dir, "schema.sql")

with open(schema_file_path, "w", encoding="utf-8") as file:
    file.write(schema_content)

print(f"💾 Task A Complete: Star Schema written perfectly to text file:\n📍 {schema_file_path}")

💾 Task A Complete: Star Schema written perfectly to text file:
📍 ..\sql\schema.sql


In [44]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import pandas as pd

# =====================================================================
# STEP 1: DEFINE TARGET SCHEMAS
# =====================================================================
table_blueprints = {

    'dim_fund': ['amfi_code', 'fund_house', 'category', 'expense_ratio','min_sip_amount', 'risk_category'],

    'dim_date': ['date_id', 'date', 'month', 'year', 'quarter', 'is_weekday'],

    'fact_nav': ['amfi_code', 'date', 'nav'],

    'fact_transactions': ['tx_d', 'investor_id', 'amfi_code', 'transaction_date', 'amount_inr', 'type', 'state', 'city', 'city_tier', 'age_group'],

    'fact_performance': ['amfi_code', 'return_1yr_pct', 'sharpe_ratio', 'alpha', 'max_drawdown_pct', 'expense_ratio_pct'],

    'fact_portfolio': ['amfi_code', 'stock_symbol', 'weight_pct', 'sector', 'portfolio_date'],

    'fact_aum': ['fund_house', 'date', 'aum_crore', 'num_schemes'],

    'fact_sip_industry': ['month', 'sip_inflow_crore', 'yoy_growth_pct', 'active_sip_accounts_crore']

}

# =====================================================================
# STEP 2: SET PATHS & RE-INITIALIZE SCHEMA
# =====================================================================
db_path = os.path.join("..", "sql", "bluestock_mf.db")
schema_path = os.path.join("..", "sql", "schema.sql")
processed_dir = os.path.join("..", "data", "processed")

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

with open(schema_path, "r", encoding="utf-8") as file:
    schema_sql = file.read()

print("🔨 Resetting database tables from schema.sql...")
with engine.begin() as connection:
    for statement in schema_sql.split(";"):
        clean_statement = statement.strip()
        if clean_statement and not clean_statement.startswith("--"):
            connection.execute(text(clean_statement))
print("✅ Base Star Schema compiled perfectly.")
print("-" * 75)

# =====================================================================
# STEP 3: SELF-HEALING INGESTION LOOP
# =====================================================================
csv_files = [f for f in os.listdir(processed_dir) if f.endswith('.csv')]
print(f"📊 Scanning {len(csv_files)} CSV files with automatic column filtering...\n")

route_summary = {table: 0 for table in table_blueprints.keys()}

for csv_name in csv_files:
    csv_path = os.path.join(processed_dir, csv_name)
    try:
        # Sneak peek at columns
        df_preview = pd.read_csv(csv_path, nrows=1)
        csv_cols_raw = list(df_preview.columns)
        csv_cols_lower = [c.lower().strip() for c in csv_cols_raw]
        
        best_table = None
        highest_score = 0
        
        # Determine the best match matching logic
        for table_name, target_cols in table_blueprints.items():
            score = len(set(csv_cols_lower).intersection(set(target_cols)))
            if score > highest_score:
                highest_score = score
                best_table = table_name
                
        if best_table and highest_score >= 1:
            # Load the full data
            df_full = pd.read_csv(csv_path)
            
            # Map raw CSV columns to their exact lowercase names to handle casing mismatches
            df_full.columns = [c.lower().strip() for c in df_full.columns]
            
            # 🚨 THE FIX: Keep ONLY columns that are explicitly expected by the destination table!
            expected_cols = table_blueprints[best_table]
            valid_cols_to_keep = [col for col in df_full.columns if col in expected_cols]
            
            # Filter the dataframe
            df_cleaned = df_full[valid_cols_to_keep].copy()
            
            # Clean up dates
            for col in df_cleaned.columns:
                if 'date' in col.lower():
                    df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce').dt.strftime('%Y-%m-%d')
            
            # Try to push to database
            df_cleaned.to_sql(best_table, con=engine, if_exists='append', index=False)
            print(f"🎯 INGESTED: '{csv_name}' ➡️ Table '{best_table}' (Kept {len(valid_cols_to_keep)} of {len(csv_cols_raw)} columns)")
            route_summary[best_table] += 1
        else:
            print(f"横️ SKIPPED : '{csv_name}' shared no structural matching attributes.")
            
    except Exception as e:
        print(f"❌ Operational Error on '{csv_name}': {e}")

print("\n" + "="*75)
print("🏁 FINAL CLEAN INGESTION REPORT")
print("="*75)
for t_name, count in route_summary.items():
    print(f"📌 Table '{t_name:<18}' loaded successfully from {count} file(s)")
print("="*75)

🔨 Resetting database tables from schema.sql...
✅ Base Star Schema compiled perfectly.
---------------------------------------------------------------------------
📊 Scanning 10 CSV files with automatic column filtering...

🎯 INGESTED: '01_fund_master_clean.csv' ➡️ Table 'dim_fund' (Kept 5 of 15 columns)
🎯 INGESTED: '03_aum_by_fund_house_clean.csv' ➡️ Table 'fact_aum' (Kept 4 of 5 columns)
🎯 INGESTED: '04_monthly_sip_inflows_clean.csv' ➡️ Table 'fact_sip_industry' (Kept 4 of 6 columns)
🎯 INGESTED: '05_category_inflows_clean.csv' ➡️ Table 'dim_fund' (Kept 1 of 3 columns)
❌ Operational Error on '06_industry_folio_count_clean.csv': (sqlite3.IntegrityError) NOT NULL constraint failed: dim_date.date
[SQL: INSERT INTO dim_date (month) VALUES (?)]
[parameters: [('2022-01',), ('2022-04',), ('2022-07',), ('2022-10',), ('2023-01',), ('2023-04',), ('2023-07',), ('2023-10',)  ... displaying 10 of 21 total bound parameter sets ...  ('2025-10',), ('2025-12',)]]
(Background on this error at: https://sq

In [6]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, inspect
import pandas as pd

# =====================================================================
# 1. SET YOUR TARGETS HERE
# =====================================================================
TARGET_TABLE = "fact_performance"          
TARGET_CSV = "clean_performance.csv" 

# =====================================================================
# 2. PATHS & ENGINE SETUP
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", TARGET_CSV))

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

# =====================================================================
# 3. RUN STRUCTURAL DIAGNOSTICS BEFORE INGESTION
# =====================================================================
print("🔍 RUNNING PRE-FLIGHT SCHEMATIC CHECKS...")
print("-" * 65)

# Check A: Verify File Existence
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Target CSV file does not exist at:\n📍 {csv_path}")

# Check B: Verify Table Existence in SQLite
inspector = inspect(engine)
existing_tables = inspector.get_table_names()

if TARGET_TABLE not in existing_tables:
    print(f"❌ ERROR: Table '{TARGET_TABLE}' does not exist inside your SQLite database file!")
    print(f"📋 Available tables in your database are: {existing_tables}")
    print("💡 Please double-check your spelling, or verify that your schema.sql successfully ran.")
else:
    print(f"✅ Database Table '{TARGET_TABLE}' found.")
    
    # Check C: Compare CSV columns against expected Database columns
    db_columns = [col['name'].lower() for col in inspector.get_columns(TARGET_TABLE)]
    df_preview = pd.read_csv(csv_path, nrows=1)
    csv_columns = [str(c).lower().strip() for c in df_preview.columns]
    
    matching_columns = [c for c in csv_columns if c in db_columns]
    
    print(f"📋 Database expects columns : {db_columns}")
    print(f"📊 CSV file has columns     : {csv_columns}")
    print(f"🎯 Overlapping match columns: {matching_columns}")
    
    if len(matching_columns) == 0:
        print("⚠️ WARNING: Zero columns match between your CSV and the target database table! Check for spelling differences.")
    
    print("-" * 65)
    
    # =====================================================================
    # 4. EXECUTE DATA LOADING PIPELINE
    # =====================================================================
    try:
        df = pd.read_csv(csv_path)
        df.columns = [c.lower().strip() for c in df.columns]
        
        # Keep only the columns present in the SQLite table schema
        df_cleaned = df[matching_columns].copy()
        
        # Format dates
        for col in df_cleaned.columns:
            if 'date' in col.lower():
                df_cleaned[col] = pd.to_datetime(df_cleaned[col], errors='coerce').dt.strftime('%Y-%m-%d')
        
        print(f"🚀 Attempting to stream data into '{TARGET_TABLE}'...")
        df_cleaned.to_sql(TARGET_TABLE, con=engine, if_exists='append', index=False)
        print(f"🎉 SUCCESS: Loaded {len(df_cleaned)} rows into '{TARGET_TABLE}' successfully.")
        
    except Exception as e:
        print("\n💥 SQLITE COMPLIANCE CRASH!")
        print(f"🛑 Detailed Error Message: {e}")

🔍 RUNNING PRE-FLIGHT SCHEMATIC CHECKS...
-----------------------------------------------------------------
✅ Database Table 'fact_performance' found.
📋 Database expects columns : ['amfi_code', 'alpha', 'sharpe_ratio']
📊 CSV file has columns     : ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'is_underperforming_risk_free']
🎯 Overlapping match columns: ['amfi_code', 'alpha', 'sharpe_ratio']
-----------------------------------------------------------------
🚀 Attempting to stream data into 'fact_performance'...
🎉 SUCCESS: Loaded 40 rows into 'fact_performance' successfully.


In [7]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import pandas as pd

# 1. Initialize Engine using our proven absolute pathing
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

with engine.begin() as connection:
    print("🔨 Adding 'daily_return_pct' column to 'fact_nav' if it doesn't exist...")
    try:
        connection.execute(text("ALTER TABLE fact_nav ADD COLUMN daily_return_pct REAL;"))
    except sa.exc.OperationalError:
        print("ℹ️ Column 'daily_return_pct' already exists. Moving to calculation phase.")

# 2. Extract pricing rows sorted chronologically to compute returns in memory
print("🔍 Fetching time-series blocks to process percentage deltas...")
query_fetch = "SELECT rowid, amfi_code, date, nav FROM fact_nav ORDER BY amfi_code, date;"
df_nav = pd.read_sql_query(query_fetch, engine)

# 3. Apply financial percentage change vector grouped by each unique fund asset
df_nav['calculated_return'] = df_nav.groupby('amfi_code')['nav'].pct_change(1) * 100
df_nav['calculated_return'] = df_nav['calculated_return'].round(4).fillna(0.0)

# 4. Write calculations back into the physical SQLite table via a transactional cursor matrix
print("🚀 Writing computed daily returns back to database table 'fact_nav'...")
update_statement = text("""
    UPDATE fact_nav 
    SET daily_return_pct = :calculated_return 
    WHERE rowid = :rowid;
""")

# Convert dataframe parameters into a list of dictionaries for high-speed batch execution
update_payload = df_nav[['calculated_return', 'rowid']].to_dict(orient='records')

with engine.begin() as connection:
    connection.execute(update_statement, update_payload)

print("✅ 'fact_nav' successfully updated with real-time historical daily returns!")

🔨 Adding 'daily_return_pct' column to 'fact_nav' if it doesn't exist...
🔍 Fetching time-series blocks to process percentage deltas...
🚀 Writing computed daily returns back to database table 'fact_nav'...
✅ 'fact_nav' successfully updated with real-time historical daily returns!


In [19]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text
import pandas as pd

# 1. Establish absolute connection paths
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

# 2. Re-create a clean structural dim_date table matching your layout
dim_date_ddl = """
DROP TABLE IF EXISTS dim_date;
CREATE TABLE dim_date (
    date_id INTEGER PRIMARY KEY AUTOINCREMENT,
    date TEXT NOT NULL UNIQUE,
    year INTEGER,
    month INTEGER,
    quarter INTEGER,
    is_weekday INTEGER
);
"""

print("🔨 Step 1: Re-compiling clean structure for 'dim_date'...")
with engine.begin() as connection:
    for statement in dim_date_ddl.strip().split(";"):
        if statement.strip():
            connection.execute(text(statement))

# 3. Pull all clean unique dates currently sitting inside fact_nav
print("🔍 Step 2: Extracting unique dates from 'fact_nav' table rows...")
unique_dates_df = pd.read_sql_query(
    "SELECT DISTINCT date FROM fact_nav WHERE date IS NOT NULL ORDER BY date;", 
    engine
)

# 4. Generate the full calendar metrics using pandas vectors
if not unique_dates_df.empty:
    print(f"🎯 Processing {len(unique_dates_df)} distinct calendar days...")
    date_series = pd.to_datetime(unique_dates_df['date'])
    
    df_calendar = pd.DataFrame()
    df_calendar['date'] = unique_dates_df['date']
    df_calendar['year'] = date_series.dt.year
    df_calendar['month'] = date_series.dt.month
    df_calendar['quarter'] = date_series.dt.quarter
    df_calendar['is_weekday'] = date_series.dt.dayofweek.apply(lambda x: 1 if x < 5 else 0)
    
    # Write back into the new dim_date structure
    df_calendar.to_sql('dim_date', con=engine, if_exists='append', index=False)
    print("✅ 'dim_date' is now built and fully synchronized with your NAV timeline history.")
else:
    print("⚠️ Process halted: 'fact_nav' returned zero date records.")

🔨 Step 1: Re-compiling clean structure for 'dim_date'...
🔍 Step 2: Extracting unique dates from 'fact_nav' table rows...
🎯 Processing 1150 distinct calendar days...
✅ 'dim_date' is now built and fully synchronized with your NAV timeline history.


In [60]:
import os
import sqlalchemy as sa
from sqlalchemy import create_engine, text, inspect
import pandas as pd

# =====================================================================
# 1. PATHS & ENGINE INITIALIZATION
# =====================================================================
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", "clean_transactions.csv"))

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))
TARGET_TABLE = "fact_transactions"

if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Could not find processed file at: {csv_path}")

try:
    # =====================================================================
    # 2. READ CSV & ANALYZE OVERLAP WITH DATABASE SCHEMA
    # =====================================================================
    print(f"📊 Reading source dataset...")
    df_tx = pd.read_csv(csv_path)
    df_tx.columns = [c.lower().strip() for c in df_tx.columns]
    
    # Auto-increment safeguard: drop tx_id from memory if it's completely empty
    if 'tx_id' in df_tx.columns and df_tx['tx_id'].isnull().all():
        df_tx = df_tx.drop(columns=['tx_id'])

    # Inspect the columns physically present in the database table right now
    inspector = inspect(engine)
    db_cols = [col['name'].lower() for col in inspector.get_columns(TARGET_TABLE)]
    
    # Calculate how many columns overlap
    matching_cols = [col for col in df_tx.columns if col in db_cols]
    print(f"🔍 Overlap Analysis: CSV shares {len(matching_cols)} columns with database table '{TARGET_TABLE}'.")

    # =====================================================================
    # 3. CONDITIONAL EXECUTION: THE "5+ COLUMNS" RULE
    # =====================================================================
    if len(matching_cols) >= 5:
        print(f"🚀 Match rate met (>= 5). Activating dynamic schema expansion...")
        
        # Find columns that are in the CSV but completely missing from the DB table
        missing_in_db = [col for col in df_tx.columns if col not in db_cols]
        
        # Safely add any new columns to the database table using ALTER TABLE
        if missing_in_db:
            print(f"🔨 Modifying database layout. Adding new columns: {missing_in_db}")
            with engine.begin() as connection:
                for new_col in missing_in_db:
                    # Deduce the approximate SQL column data type based on Pandas values
                    if pd.api.types.is_numeric_dtype(df_tx[new_col]):
                        col_type = "REAL"
                    else:
                        col_type = "TEXT"
                    
                    alter_query = f"ALTER TABLE {TARGET_TABLE} ADD COLUMN {new_col} {col_type};"
                    connection.execute(text(alter_query))
            print("✅ Database layout successfully expanded.")
        
        # Clear previous records to keep data fresh
        with engine.begin() as connection:
            connection.execute(text(f"DELETE FROM {TARGET_TABLE};"))
            
        # Stream the ENTIRE dataset cleanly now that the database accommodates it
        print(f"🚀 Streaming all {len(df_tx)} columns and rows into '{TARGET_TABLE}'...")
        df_tx.to_sql(TARGET_TABLE, con=engine, if_exists='append', index=False)
        print("🎉 SUCCESS: Entire CSV loaded perfectly with an expanded schema configuration!")
        
    else:
        print(f"⚠️ SKIPPED: The CSV only shared {len(matching_cols)} columns (Threshold requires >= 5).")

    # =====================================================================
    # 5. VERIFY FINAL PHYSICAL DATABASE COLUMNS
    # =====================================================================
    updated_db_cols = [col['name'] for col in inspect(engine).get_columns(TARGET_TABLE)]
    print("-" * 75)
    print(f"📋 FINAL PHYSICAL DATABASE COLUMNS ON DISK: {updated_db_cols}")
    print("-" * 75)

except Exception as e:
    print(f"❌ Schema-Padding Ingestion Failed: {e}")

📊 Reading source dataset...
🔍 Overlap Analysis: CSV shares 10 columns with database table 'fact_transactions'.
🚀 Match rate met (>= 5). Activating dynamic schema expansion...
🔨 Modifying database layout. Adding new columns: ['annual_income_lakh', 'payment_mode', 'kyc_status']
✅ Database layout successfully expanded.
🚀 Streaming all 32778 columns and rows into 'fact_transactions'...
🎉 SUCCESS: Entire CSV loaded perfectly with an expanded schema configuration!
---------------------------------------------------------------------------
📋 FINAL PHYSICAL DATABASE COLUMNS ON DISK: ['tx_id', 'investor_id', 'amfi_code', 'transaction_date', 'amount_inr', 'transaction_type', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']
---------------------------------------------------------------------------


In [61]:
from sqlalchemy import inspect

inspector = inspect(engine)
real_columns = [col['name'] for col in inspector.get_columns('fact_transactions')]

print("=" * 50)
print(f"📋 ACTUAL COLUMNS INSIDE 'fact_transactions': {real_columns}")
print("=" * 50)

📋 ACTUAL COLUMNS INSIDE 'fact_transactions': ['tx_id', 'investor_id', 'amfi_code', 'transaction_date', 'amount_inr', 'transaction_type', 'state', 'city', 'city_tier', 'age_group', 'gender', 'annual_income_lakh', 'payment_mode', 'kyc_status']


In [26]:
from sqlalchemy import inspect

inspector = inspect(engine)
real_columns = [col['name'] for col in inspector.get_columns('fact_performance')]

print("=" * 50)
print(f"📋 ACTUAL COLUMNS INSIDE 'fact_performance': {real_columns}")
print("=" * 50)

📋 ACTUAL COLUMNS INSIDE 'fact_performance': ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'is_underperforming_risk_free']


In [25]:
# 1. Map to your performance target CSV file
csv_filename = "clean_performance.csv" 
csv_path = os.path.abspath(os.path.join(base_dir, "data", "processed", csv_filename))

print(f"\n🔍 Task 2: Reading structural columns from file: {csv_filename}...")
if not os.path.exists(csv_path):
    raise FileNotFoundError(f"❌ Could not find processed file at: {csv_path}")

# Load full dataset and lower column names to enforce consistency
df_perf = pd.read_csv(csv_path)
df_perf.columns = [c.lower().strip() for c in df_perf.columns]
csv_cols = list(df_perf.columns)
print(f"📋 Found columns in CSV: {csv_cols}")

# 2. Drop the old mismatched table and build a fresh schema dynamically based on the file headers
print("🔨 Dropping old table and dynamically building fresh layout for 'fact_performance'...")

# We map Pandas data types to generic SQL columns
sql_types = []
for col in csv_cols:
    if col == "amfi_code":
        sql_types.append(f"{col} INTEGER")
    elif "ratio" in col or "pct" in col or "alpha" in col or "return" in col or "drawdown" in col:
        sql_types.append(f"{col} REAL")
    else:
        sql_types.append(f"{col} TEXT")

# Append your required foreign key link back to dim_fund
columns_payload = ", ".join(sql_types)
dynamic_ddl = f"""
DROP TABLE IF EXISTS fact_performance;
CREATE TABLE fact_performance (
    {columns_payload},
    FOREIGN KEY (amfi_code) REFERENCES dim_fund(amfi_code)
);
"""

with engine.begin() as connection:
    for statement in dynamic_ddl.strip().split(";"):
        if statement.strip():
            connection.execute(text(statement))
print("✅ New 'fact_performance' table compiled successfully on disk.")

# 3. Stream data into the freshly built database table
print(f"🚀 Streaming {len(df_perf)} data records into 'fact_performance'...")
df_perf.to_sql('fact_performance', con=engine, if_exists='append', index=False)
print("🎉 Success: 'fact_performance' table is fixed and fully populated!")

# 4. View final confirmation snapshot summary frame
print("\n📋 QUICK DATABASE PREVIEW SNAPSHOT:")
print("-" * 65)
print(pd.read_sql_query("SELECT * FROM fact_performance LIMIT 3;", engine))


🔍 Task 2: Reading structural columns from file: clean_performance.csv...
📋 Found columns in CSV: ['amfi_code', 'scheme_name', 'fund_house', 'category', 'plan', 'return_1yr_pct', 'return_3yr_pct', 'return_5yr_pct', 'benchmark_3yr_pct', 'alpha', 'beta', 'sharpe_ratio', 'sortino_ratio', 'std_dev_ann_pct', 'max_drawdown_pct', 'aum_crore', 'expense_ratio_pct', 'morningstar_rating', 'risk_grade', 'is_underperforming_risk_free']
🔨 Dropping old table and dynamically building fresh layout for 'fact_performance'...
✅ New 'fact_performance' table compiled successfully on disk.
🚀 Streaming 40 data records into 'fact_performance'...
🎉 Success: 'fact_performance' table is fixed and fully populated!

📋 QUICK DATABASE PREVIEW SNAPSHOT:
-----------------------------------------------------------------
   amfi_code                                 scheme_name       fund_house  \
0     119551   SBI Bluechip Fund - Regular Plan - Growth  SBI Mutual Fund   
1     119552    SBI Bluechip Fund - Direct Plan -

In [66]:
import os
import sqlalchemy as sa
import pandas as pd

# 1. Establish absolute connection path
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))
# Query-1: Write the analytical aggregation query
top_5_aum_query = """
SELECT 
    f.fund_house,
    MAX(a.aum_crore) AS total_aum_crore,
    a.num_schemes,
    a.date AS reporting_date
FROM fact_aum a
JOIN dim_fund f ON a.fund_house = f.fund_house
GROUP BY f.fund_house
ORDER BY total_aum_crore DESC
LIMIT 5;
"""

# 3. Execute and display the data matrix
try:
    top_5_df = pd.read_sql_query(sa.text(top_5_aum_query), engine)
    print(" TOP 5 MUTUAL FUND HOUSES BY AUM MASS SCALE")
    print("-" * 65)
    display(top_5_df)
except Exception as e:
    print(f"Execution Error: {e}")


# Query-2: Write the monthly pricing aggregation query using structural joins
avg_nav_query = """
SELECT 
    n.amfi_code,
    d.year,
    d.month,
    ROUND(AVG(n.nav), 4) AS avg_monthly_nav,
    COUNT(n.nav) AS trading_days_logged
FROM fact_nav n
JOIN dim_date d ON n.date = d.date
GROUP BY n.amfi_code, d.year, d.month
ORDER BY n.amfi_code ASC, d.year DESC, d.month DESC;
"""

# 3. Execute and display the data matrix
try:
    avg_nav_df = pd.read_sql_query(sa.text(avg_nav_query), engine)
    print("📈 ASSET PRICING EVOLUTION: AVERAGE NAV PER MONTH")
    print("-" * 65)
    
    # Using head(20) to show a clean sample layout without flooding your screen
    display(avg_nav_df.head(20))
    print(f"\n Total grouped monthly records computed: {len(avg_nav_df)}")
except Exception as e:
    print(f" Execution Error: {e}")


# Query-3: Write the YoY Growth Analysis Query
sip_industry_query = """
WITH AnnualizedIndustrySIP AS (
    SELECT 
        -- If your month is stored as an integer like 202501, we extract the first 4 digits as the year
        CASE 
            WHEN LENGTH(CAST(month AS TEXT)) >= 4 THEN CAST(SUBSTR(CAST(month AS TEXT), 1, 4) AS INTEGER)
            ELSE month 
        END AS calendar_year,
        ROUND(SUM(sip_inflow_crore), 2) AS total_year_inflow_crore,
        ROUND(AVG(active_sip_accounts_crore), 2) AS avg_active_accounts_crore
    FROM fact_sip_industry
    GROUP BY calendar_year
)
SELECT 
    calendar_year,
    total_year_inflow_crore,
    -- Pull the previous year's aggregated totals
    LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year) AS prior_year_inflow_crore,
    -- Calculate the exact YoY change percentage margin
    ROUND(
        ((total_year_inflow_crore - LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year)) / 
        LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year)) * 100, 
        2
    ) AS yoy_growth_pct,
    avg_active_accounts_crore
FROM AnnualizedIndustrySIP
ORDER BY calendar_year DESC;
"""

# 3. Execute and display the results
try:
    sip_industry_df = pd.read_sql_query(sa.text(sip_industry_query), engine)
    print("📈 AMFI MACRO LABELS: SIP INFLOW YEAR-OVER-YEAR (YoY) GROWTH TRENDS")
    print("-" * 75)
    display(sip_industry_df)
except Exception as e:
    print(f" Execution Error: {e}")


# Query-4: Write the geographicaggregation query
state_transactions_query = """
SELECT 
    state AS geographic_state,
    COUNT(tx_id) AS total_transactions_executed,
    ROUND(SUM(amount_inr), 2) AS total_capital_volume_crore,
    ROUND(AVG(amount_inr), 2) AS average_transaction_size
FROM fact_transactions
WHERE state IS NOT NULL
GROUP BY state
ORDER BY total_capital_volume_crore DESC;
"""

try:
    state_df = pd.read_sql_query(sa.text(state_transactions_query), engine)
    print(" GEOGRAPHIC DISTRIBUTION: TRANSACTIONS BY STATE")
    print("-" * 75)
    display(state_df)
except Exception as e:
    print(f" Execution Error: {e}")


# Query-5: Write the structural filtering query
# Filtering where expense ratio (expense_ratio) is strictly less than 1.0
low_expense_query = """
SELECT 
    amfi_code,
    fund_house,
    category,
    expense_ratio_pct AS expense_ratio
FROM dim_fund
WHERE expense_ratio < 1.0 AND expense_ratio IS NOT NULL
ORDER BY expense_ratio ASC;
"""

# 3. Execute and display the results matrix
try:
    low_expense_df = pd.read_sql_query(sa.text(low_expense_query), engine)
    print(" COST-EFFICIENT ASSETS: FUNDS WITH EXPENSE RATIO < 1%")
    print("-" * 75)
    
    # Display the results
    display(low_expense_df)
    print(f"\n🎯 Total low-cost fund variants discovered: {len(low_expense_df)}")
except Exception as e:
    print(f" Execution Error: {e}")

 TOP 5 MUTUAL FUND HOUSES BY AUM MASS SCALE
-----------------------------------------------------------------


,fund_house,total_aum_crore,num_schemes,reporting_date
0,SBI Mutual Fund,1250000,186,2025-03-31
1,ICICI Prudential MF,1074000,216,2025-12-31
2,HDFC Mutual Fund,930000,195,2025-12-31
3,Nippon India MF,700000,177,2025-12-31
4,Kotak Mahindra MF,580000,168,2025-12-31


📈 ASSET PRICING EVOLUTION: AVERAGE NAV PER MONTH
-----------------------------------------------------------------


,amfi_code,year,month,avg_monthly_nav,trading_days_logged
0,100016,2026,5,592.4348,189
1,100016,2026,4,610.2515,198
2,100016,2026,3,588.1440,198
3,100016,2026,2,593.4026,180
4,100016,2026,1,632.3917,198
5,100016,2025,12,618.9595,207
6,100016,2025,11,601.5346,180
7,100016,2025,10,616.8069,207
8,100016,2025,9,594.9615,198
9,100016,2025,8,590.5114,189



 Total grouped monthly records computed: 2120
📈 AMFI MACRO LABELS: SIP INFLOW YEAR-OVER-YEAR (YoY) GROWTH TRENDS
---------------------------------------------------------------------------


...

 GEOGRAPHIC DISTRIBUTION: TRANSACTIONS BY STATE
---------------------------------------------------------------------------


,geographic_state,total_transactions_executed,total_capital_volume_crore,average_transaction_size
0,Punjab,2965,315780459.0,106502.68
1,Tamil Nadu,2806,315177237.0,112322.61
2,Madhya Pradesh,2931,308312493.0,105190.21
3,Rajasthan,2577,298645822.0,115888.95
4,Gujarat,2780,298358940.0,107323.36
5,West Bengal,2748,297182514.0,108145.02
6,Telangana,2718,290219284.0,106776.78
7,Delhi,2677,289633404.0,108193.28
8,Uttar Pradesh,2695,285368873.0,105888.26
9,Haryana,2736,279634354.0,102205.54


 COST-EFFICIENT ASSETS: FUNDS WITH EXPENSE RATIO < 1%
---------------------------------------------------------------------------


,amfi_code,fund_house,category,expense_ratio
0,118636,Nippon India MF,Debt,0.55
1,100025,HDFC Mutual Fund,Debt,0.56
2,120844,Kotak Mahindra MF,Debt,0.60
3,119552,SBI Mutual Fund,Equity,0.66
4,119599,SBI Mutual Fund,Equity,0.72
5,118633,Nippon India MF,Equity,0.72
6,120507,ICICI Prudential MF,Debt,0.74
7,119093,Axis Mutual Fund,Equity,0.75
8,119120,SBI Mutual Fund,Debt,0.77
9,125498,HDFC Mutual Fund,Equity,0.78



🎯 Total low-cost fund variants discovered: 14


In [73]:
import os
import sqlalchemy as sa
import pandas as pd

# 1. Establish absolute connection path
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

## Query-6:  Write the optimized data-audit query
# This finds funds in the catalog that have 0 records inside fact_nav
zombie_funds_query = """
SELECT 
    f.amfi_code,
    f.fund_house,
    f.category
FROM dim_fund f
WHERE NOT EXISTS (
    SELECT 1 
    FROM fact_nav n 
    WHERE n.amfi_code = f.amfi_code
);
"""

# 3. Execute and display the data-quality report
try:
    zombie_df = pd.read_sql_query(sa.text(zombie_funds_query), engine)
    print("🔍 DATA QUALITY AUDIT: IDENTIFYING DORMANT (ZOMBIE) FUNDS")
    print("-" * 75)
    
    if zombie_df.empty:
        print(" Excellent! Zero zombie funds found. Every fund in your catalog has active NAV data rows.")
    else:
        print(f" Alert: Found {len(zombie_df)} funds with completely missing time-series history.")
        display(zombie_df)
        
except Exception as e:
    print(f" Execution Error: {e}")


# Query-7: Write the categorical profiling query
category_distribution_query = """
SELECT 
    category AS fund_category, 
    COUNT(amfi_code) AS total_funds_count
FROM dim_fund
WHERE category IS NOT NULL AND category != ''
GROUP BY category
ORDER BY total_funds_count DESC;
"""

# 3. Execute and display the breakdown table
try:
    category_df = pd.read_sql_query(sa.text(category_distribution_query), engine)
    print(" CATALOG PROFILING: FUND DISTRIBUTION BY CATEGORY")
    print("-" * 65)
    
    if category_df.empty:
        print(" Notice: The query returned no results. Ensure your 'dim_fund' table contains category labels.")
    else:
        display(category_df)
        
        # Quick bonus: Calculate the total inventory size dynamically
        print(f"\n Total unique schemes indexed in catalog: {category_df['total_funds_count'].sum()}")
        
except Exception as e:
    print(f" Execution Error: {e}")


# Query-8: Write the capital outlier transaction query
whale_tx_query = """
SELECT 
    investor_id, 
    amfi_code, 
    transaction_date, 
    amount_inr AS transaction_amount_crore, 
    transaction_type AS transaction_type, 
    state AS allocation_state
FROM fact_transactions
WHERE amount_inr > 5.0 AND transaction_type = 'LUMPSUM'
ORDER BY amount_inr DESC;
"""

# 3. Execute and display the financial ledger matrix
try:
    whale_df = pd.read_sql_query(sa.text(whale_tx_query), engine)
    print(" RISK ANALYSIS: HIGH-VALUE 'WHALE' LUMPSUM TRANSACTIONS (> 5 CRORE)")
    print("-" * 80)
    
    if whale_df.empty:
        print(" Note: No individual transaction exceeded 5 Crore. Try lowering the threshold to 'amount > 1.0' to catch smaller whales.")
    else:
        display(whale_df)
        print(f"\n🎯 Total high-net-worth liquidity alerts captured: {len(whale_df)}")
        
except Exception as e:
    print(f" Execution Error: {e}")


# Query-9: Write the absolute variance volatility query
extreme_returns_query = """
SELECT 
    amfi_code, 
    date, 
    nav AS closing_nav, 
    daily_return_pct
FROM fact_nav
WHERE daily_return_pct IS NOT NULL AND daily_return_pct != 0.0
ORDER BY ABS(daily_return_pct) DESC
LIMIT 10;
"""

# 3. Execute and display the volatility matrix
try:
    volatility_df = pd.read_sql_query(sa.text(extreme_returns_query), engine)
    print(" QUANT RISK RADAR: TOP 10 MOST VOLATILE HISTORICAL TRADING ENTRIES")
    print("-" * 80)
    
    if volatility_df.empty:
        print(" Note: No non-zero daily returns found. Ensure 'daily_return_pct' was calculated and populated.")
    else:
        display(volatility_df)
except Exception as e:
    print(f" Execution Error: {e}")


# Query-10: Write the calendar baseline verification query
weekend_vs_weekday_query = """
SELECT 
    CASE 
        WHEN is_weekday = 1 THEN 'Active Trading Weekdays'
        WHEN is_weekday = 0 THEN 'Weekend Market Close Gaps'
        ELSE 'Unknown/Miscalibrated'
    END AS day_type_classification,
    COUNT(date_id) AS total_calendar_days_dimensioned
FROM dim_date
GROUP BY is_weekday;
"""

# 3. Execute and display the temporal breakdown framework
try:
    calendar_df = pd.read_sql_query(sa.text(weekend_vs_weekday_query), engine)
    print(" TIMELINE HEALTH AUDIT: WEEKDAY VS WEEKEND RECONCILIATION")
    print("-" * 75)
    
    if calendar_df.empty:
        print(" Alert: The 'dim_date' table is currently empty. Run your date-seeding pipeline first.")
    else:
        display(calendar_df)
        
        # Calculate proportional distribution percentages for the data profiling summary
        total_days = calendar_df['total_calendar_days_dimensioned'].sum()
        print(f"\n Comprehensive Timeline Depth: {total_days} total sequential days tracked.")
        
except Exception as e:
    print(f" Execution Error: {e}")

🔍 DATA QUALITY AUDIT: IDENTIFYING DORMANT (ZOMBIE) FUNDS
---------------------------------------------------------------------------
 Alert: Found 1152 funds with completely missing time-series history.


,amfi_code,fund_house,category
0,None,None,Large Cap
1,None,None,Mid Cap
2,None,None,Small Cap
3,None,None,Flexi Cap
4,None,None,Large & Mid Cap
...,...,...,...
1147,None,None,Sectoral/Thematic
1148,None,None,Liquid
1149,None,None,Short Duration
1150,None,None,Gilt


 CATALOG PROFILING: FUND DISTRIBUTION BY CATEGORY
-----------------------------------------------------------------


,fund_category,total_funds_count
0,Equity,306
1,Debt,54
2,Large Cap,14
3,Mid Cap,7
4,Small Cap,6
5,Liquid,3
6,Gilt,2
7,Flexi Cap,2
8,Value,1
9,Short Duration,1



 Total unique schemes indexed in catalog: 400
 RISK ANALYSIS: HIGH-VALUE 'WHALE' LUMPSUM TRANSACTIONS (> 5 CRORE)
--------------------------------------------------------------------------------


...


🎯 Total high-net-worth liquidity alerts captured: 8095
 QUANT RISK RADAR: TOP 10 MOST VOLATILE HISTORICAL TRADING ENTRIES
--------------------------------------------------------------------------------


,amfi_code,date,closing_nav,daily_return_pct
0,119598,2024-04-15,189.0742,6.4713
1,118634,2024-03-19,110.5401,5.9304
2,118634,2022-06-24,67.2534,-5.8102
3,101207,2024-05-01,65.5882,5.4851
4,119599,2023-01-10,153.2221,5.3320
5,101207,2023-03-22,56.8326,-5.1847
6,119599,2024-01-17,136.8239,5.1811
7,119598,2022-01-10,96.0964,5.1113
8,118634,2023-03-01,113.5801,-5.0335
9,119598,2022-11-08,123.9704,4.9051


 TIMELINE HEALTH AUDIT: WEEKDAY VS WEEKEND RECONCILIATION
---------------------------------------------------------------------------


,day_type_classification,total_calendar_days_dimensioned
0,Active Trading Weekdays,1150



 Comprehensive Timeline Depth: 1150 total sequential days tracked.


In [76]:
import os
import sqlalchemy as sa
import pandas as pd

# 1. Establish absolute directory path connections
base_dir = os.path.dirname(os.getcwd())
db_path = os.path.abspath(os.path.join(base_dir, "sql", "bluestock_mf.db"))
queries_file_path = os.path.abspath(os.path.join(base_dir, "sql", "queries.sql"))

engine = create_engine(sa.URL.create(drivername="sqlite", database=db_path))

# 2. Map all 10 verified core analytical query structures into a dictionary registry
query_registry = {
    "QUERY 1: Macro Leaderboard - Top 5 Fund Houses by Corporate AUM": """
        SELECT f.fund_house, MAX(a.aum_crore) AS total_aum_crore, a.num_schemes, a.date AS reporting_date
        FROM fact_aum a
        JOIN dim_fund f ON a.fund_house = f.fund_house
        GROUP BY f.fund_house ORDER BY total_aum_crore DESC LIMIT 5;
    """,
    "QUERY 2: Asset Pricing Evolution - Average NAV Per Month": """
        SELECT n.amfi_code, d.year, d.month, ROUND(AVG(n.nav), 4) AS avg_monthly_nav, COUNT(n.nav) AS trading_days_logged
        FROM fact_nav n JOIN dim_date d ON n.date = d.date
        GROUP BY n.amfi_code, d.year, d.month ORDER BY n.amfi_code ASC, d.year DESC, d.month DESC LIMIT 15;
    """,
    "QUERY 3: Macro Retail Velocity - AMFI Industry SIP Inflow YoY Growth": """
        WITH AnnualizedIndustrySIP AS (
            SELECT CASE WHEN LENGTH(CAST(month AS TEXT)) >= 4 THEN CAST(SUBSTR(CAST(month AS TEXT), 1, 4) AS INTEGER) ELSE month END AS calendar_year,
                   ROUND(SUM(sip_inflow_crore), 2) AS total_year_inflow_crore, ROUND(AVG(active_sip_accounts_crore), 2) AS avg_active_accounts_crore
            FROM fact_sip_industry GROUP BY calendar_year
        )
        SELECT calendar_year, total_year_inflow_crore, LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year) AS prior_year_inflow_crore,
               ROUND(((total_year_inflow_crore - LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year)) / LAG(total_year_inflow_crore, 1) OVER (ORDER BY calendar_year)) * 100, 2) AS yoy_growth_pct
        FROM AnnualizedIndustrySIP ORDER BY calendar_year DESC;
    """,
    "QUERY 4: Geographic Capital Densities - Transactions and Volume by State": """
        SELECT state AS geographic_state, COUNT(tx_id) AS total_transactions_executed, ROUND(SUM(amount_inr), 2) AS total_capital_volume_crore
        FROM fact_transactions WHERE state IS NOT NULL GROUP BY state ORDER BY total_capital_volume_crore DESC;
    """,
    "QUERY 5: Cost-Efficient Asset Screening - Expense Ratio Under 1%": """
        SELECT amfi_code, fund_house, category, expense_ratio_pct AS expense_ratio_pct FROM dim_fund
        WHERE expense_ratio_pct < 1.0 AND expense_ratio_pct IS NOT NULL ORDER BY expense_ratio_pct ASC LIMIT 15;
    """,
    "QUERY 6: Data Quality Audit - Identifying Dormant (Zombie) Funds": """
        SELECT f.amfi_code, f.fund_house, f.category FROM dim_fund f
        WHERE NOT EXISTS (SELECT 1 FROM fact_nav n WHERE n.amfi_code = f.amfi_code);
    """,
    "QUERY 7: Catalog Structural Profiling - Inventory Share by Category": """
        SELECT category AS fund_category, COUNT(amfi_code) AS total_funds_count FROM dim_fund
        WHERE category IS NOT NULL AND category != '' GROUP BY category ORDER BY total_funds_count DESC;
    """,
    "QUERY 8: Risk Vulnerability Radar - High-Value Lumpsum 'Whale' Logs (> 5Cr)": """
        SELECT investor_id, amfi_code, transaction_date, amount_inr AS transaction_amount_crore, transaction_type AS transaction_type FROM fact_transactions
        WHERE amount_inr > 5.0 AND transaction_type = 'LUMPSUM' ORDER BY amount_inr DESC LIMIT 15;
    """,
    "QUERY 9: Quant Volatility Shock Tracking - Top 10 Extreme Return Days": """
        SELECT amfi_code, date, nav AS closing_nav, daily_return_pct FROM fact_nav
        WHERE daily_return_pct IS NOT NULL AND daily_return_pct != 0.0 ORDER BY ABS(daily_return_pct) DESC LIMIT 10;
    """,
    "QUERY 10: Timeline Calibration Audit - Business Weekday vs Weekend Gap Metrics": """
        SELECT CASE WHEN is_weekday = 1 THEN 'Active Trading Weekdays' WHEN is_weekday = 0 THEN 'Weekend Market Close Gaps' ELSE 'Unknown' END AS day_type_classification,
               COUNT(date_id) AS total_calendar_days_dimensioned FROM dim_date GROUP BY is_weekday;
    """
}

# 3. Build the core output payload content
with open(queries_file_path, "w", encoding="utf-8") as file:
    file.write("""-- ====================================================================
-- COMPREHENSIVE PRODUCTION ANALYTICS WORKBOOK & OUTPUT LOG
-- PROJECT: MUTUAL FUNDS ANALYTICS PLATFORM
-- GENERATED: SYSTEM DYNAMIC PIPELINE VERIFICATION RUN
-- ====================================================================
\n""")

    for title, query in query_registry.items():
        print(f" Running execution loop for: {title}...")
        
        # Write the SQL query syntax out to the file first
        file.write(f"-- {'=' * 76}\n")
        file.write(f"-- {title}\n")
        file.write(f"-- {'=' * 76}\n")
        file.write(query.strip() + ";\n\n")
        
        # Execute query and formatting snapshot data rows
        try:
            query_result_df = pd.read_sql_query(sa.text(query), engine)
            
            file.write("--  REPOSITORY LIVE DATA SNAPSHOT OUTPUT:\n")
            if query_result_df.empty:
                file.write("-- [Empty Dataset Matrix: Zero Rows Returned]\n\n")
            else:
                # Convert the dataframe to a clean markdown table string, prefixed with SQL comment dashes
                markdown_table = query_result_df.to_markdown(index=False)
                commented_table = "\n".join([f"-- {line}" for line in markdown_table.split("\n")])
                file.write(commented_table + "\n\n")
                
        except Exception as query_error:
            file.write(f"--  RUNTIME EXECUTION ERROR:\n-- {str(query_error)}\n\n")

print(f"\n💾 Success! Fully executed data log saved into 'queries.sql'.")
print(f"📍 Location Path: {queries_file_path}")

 Running execution loop for: QUERY 1: Macro Leaderboard - Top 5 Fund Houses by Corporate AUM...
 Running execution loop for: QUERY 2: Asset Pricing Evolution - Average NAV Per Month...
 Running execution loop for: QUERY 3: Macro Retail Velocity - AMFI Industry SIP Inflow YoY Growth...
 Running execution loop for: QUERY 4: Geographic Capital Densities - Transactions and Volume by State...
 Running execution loop for: QUERY 5: Cost-Efficient Asset Screening - Expense Ratio Under 1%...
 Running execution loop for: QUERY 6: Data Quality Audit - Identifying Dormant (Zombie) Funds...
 Running execution loop for: QUERY 7: Catalog Structural Profiling - Inventory Share by Category...
 Running execution loop for: QUERY 8: Risk Vulnerability Radar - High-Value Lumpsum 'Whale' Logs (> 5Cr)...
 Running execution loop for: QUERY 9: Quant Volatility Shock Tracking - Top 10 Extreme Return Days...
 Running execution loop for: QUERY 10: Timeline Calibration Audit - Business Weekday vs Weekend Gap Metri